# Relative Wealth Index (Brazil) - Transformation (v1)

This notebook converts `data/bra_relative_wealth_index.csv` into:
- GeoTIFF in EPSG:4326 (gridded from points)
- COG in EPSG:3857
- Visual XYZ tiles
- Value XYZ tiles (RGB packed)
- Metadata JSON for frontend decoding and attribution

Attribution (required):

Microestimates of wealth for all low- and middle-income countries.
Guanghua Chi, Han Fang, Sourav Chatterjee, Joshua E. Blumenstock.
Proceedings of the National Academy of Sciences Jan 2022, 119 (3) e2113658119.
DOI: 10.1073/pnas.2113658119

In [1]:
import csv
import json
import shutil
import subprocess
from pathlib import Path

BASE = Path('.')
DATA_DIR = BASE / 'data'
OUT_DIR = BASE / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / 'bra_relative_wealth_index.csv'
VRT_PATH = DATA_DIR / 'bra_relative_wealth_index.vrt'

RWI_4326 = OUT_DIR / 'bra_relative_wealth_index_4326.tif'
RWI_3857 = OUT_DIR / 'bra_relative_wealth_index_3857.tif'
RWI_COG = OUT_DIR / 'bra_relative_wealth_index_cog.tif'

RWI_COLORIZED = OUT_DIR / 'bra_relative_wealth_index_colorized.tif'
RWI_VALUE_SCALED = OUT_DIR / 'bra_relative_wealth_index_scaled_int.tif'
RWI_VALUE_ENCODED = OUT_DIR / 'bra_relative_wealth_index_value_encoded.tif'

TILES_VISUAL = OUT_DIR / 'tiles_visual'
TILES_VALUES = OUT_DIR / 'tiles_values'

VISUAL_COLORS = DATA_DIR / 'rwi_visual_colors.txt'
VALUE_COLORS = DATA_DIR / 'rwi_value_encoding_colors.txt'
METADATA_JSON = OUT_DIR / 'metadata.json'

RWI_OFFSET = 2.0
RWI_SCALE = 1000.0
RWI_UNIT = 'relative_wealth_index'
RWI_NODATA = -9999.0

if not CSV_PATH.exists():
    raise FileNotFoundError(f'Missing source CSV: {CSV_PATH}')

print('Input CSV:', CSV_PATH)

Input CSV: data/bra_relative_wealth_index.csv


In [2]:
# Inspect CSV and derive grid specs
min_lat, max_lat = float('inf'), float('-inf')
min_lon, max_lon = float('inf'), float('-inf')
min_rwi, max_rwi = float('inf'), float('-inf')

lat_vals = set()
lon_vals = set()
rows = 0

with CSV_PATH.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        lat = float(row['latitude'])
        lon = float(row['longitude'])
        rwi = float(row['rwi'])

        min_lat = min(min_lat, lat)
        max_lat = max(max_lat, lat)
        min_lon = min(min_lon, lon)
        max_lon = max(max_lon, lon)
        min_rwi = min(min_rwi, rwi)
        max_rwi = max(max_rwi, rwi)

        lat_vals.add(lat)
        lon_vals.add(lon)
        rows += 1

lat_sorted = sorted(lat_vals)
lon_sorted = sorted(lon_vals)
ny = len(lat_sorted)
nx = len(lon_sorted)

# Include half-cell margins so points represent cell centers
lat_step = min(abs(lat_sorted[i + 1] - lat_sorted[i]) for i in range(len(lat_sorted) - 1))
lon_step = min(abs(lon_sorted[i + 1] - lon_sorted[i]) for i in range(len(lon_sorted) - 1))

txe_min = min_lon - lon_step / 2.0
txe_max = max_lon + lon_step / 2.0
tye_min = min_lat - lat_step / 2.0
tye_max = max_lat + lat_step / 2.0

print('Rows:', rows)
print('Grid size (nx, ny):', nx, ny)
print('Lon range:', (min_lon, max_lon), 'step:', lon_step)
print('Lat range:', (min_lat, max_lat), 'step:', lat_step)
print('RWI range:', (min_rwi, max_rwi))

Rows: 173473
Grid size (nx, ny): 1740 1785
Lon range: (-73.6633301, -34.7937012) step: 0.02197259999999801
Lat range: (-33.7334762, 5.0362268) step: 0.0182749999999956
RWI range: (-1.72, 2.088)


In [3]:
# Build GDAL VRT for lon/lat point geometry
vrt_xml = f'''<OGRVRTDataSource>
  <OGRVRTLayer name="bra_relative_wealth_index">
    <SrcDataSource>{CSV_PATH.as_posix()}</SrcDataSource>
    <GeometryType>wkbPoint</GeometryType>
    <LayerSRS>EPSG:4326</LayerSRS>
    <GeometryField encoding="PointFromColumns" x="longitude" y="latitude"/>
  </OGRVRTLayer>
</OGRVRTDataSource>
'''
VRT_PATH.write_text(vrt_xml, encoding='utf-8')
print('Wrote VRT:', VRT_PATH)

Wrote VRT: data/bra_relative_wealth_index.vrt


In [4]:
# Grid points to raster in EPSG:4326 (nearest assignment)
subprocess.run([
    'gdal_grid',
    '-a', f'nearest:radius1={lon_step}:radius2={lat_step}:angle=0:nodata={RWI_NODATA}',
    '-zfield', 'rwi',
    '-txe', str(txe_min), str(txe_max),
    '-tye', str(tye_min), str(tye_max),
    '-outsize', str(nx), str(ny),
    '-a_srs', 'EPSG:4326',
    '-ot', 'Float32',
    '-of', 'GTiff',
    str(VRT_PATH),
    str(RWI_4326)
], check=True)
print('Created 4326 raster:', RWI_4326)

Grid data type is "Float32"
Grid size = (1740 1785).
Corner coordinates = (-73.674316 5.045364)-(-34.782715 -33.742614).
Grid cell size = (0.022351 -0.021730).
Source point count = 173473.
Algorithm name: "nearest".
Options are "radius1=0.021973:radius2=0.018275:angle=0.000000:nodata=-9999.000000"

0...10...20...30...40...50...60...70...80...90...100 - done.
Created 4326 raster: output/bra_relative_wealth_index_4326.tif


In [5]:
# Reproject to Web Mercator and convert to COG
subprocess.run([
    'gdalwarp',
    '-t_srs', 'EPSG:3857',
    '-r', 'bilinear',
    '-dstnodata', str(RWI_NODATA),
    str(RWI_4326),
    str(RWI_3857)
], check=True)

subprocess.run([
    'gdal_translate',
    str(RWI_3857),
    str(RWI_COG),
    '-of', 'COG',
    '-co', 'COMPRESS=DEFLATE',
    '-co', 'OVERVIEWS=AUTO',
    '-co', 'RESAMPLING=BILINEAR'
], check=True)
print('Created COG:', RWI_COG)

Using internal nodata values (e.g. -9999) for image output/bra_relative_wealth_index_4326.tif.
Processing bra_relative_wealth_index_4326.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 1717, 1807
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: output/bra_relative_wealth_index_cog.tif


In [ ]:
# Visual color ramp and visual tiles
visual_stops = [
    (-1.8, (49, 54, 149)),
    (-1.0, (69, 117, 180)),
    (-0.3, (116, 173, 209)),
    (0.0, (224, 243, 248)),
    (0.6, (171, 221, 164)),
    (1.2, (253, 174, 97)),
    (2.1, (215, 25, 28)),
]

with VISUAL_COLORS.open('w', encoding='utf-8') as f:
    f.write('nv 0 0 0 0\n')
    for v, (r, g, b) in visual_stops:
        f.write(f'{v:.3f} {r} {g} {b}\n')

subprocess.run([
    'gdaldem', 'color-relief',
    str(RWI_COG),
    str(VISUAL_COLORS),
    str(RWI_COLORIZED)
], check=True)

TILES_VISUAL.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'gdal2tiles.py', '-r', 'bilinear', '-z', '6-14', '--xyz', '-w', 'none',
    str(RWI_COLORIZED), str(TILES_VISUAL)
], check=True)
print('Visual tiles written to:', TILES_VISUAL)

0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0

In [ ]:
# Value tiles: scale RWI to UInt16 then encode integer value to RGB
# decode formula: rwi = (R + 256*G + 65536*B) / scale - offset
scaled_min = int(round((min_rwi + RWI_OFFSET) * RWI_SCALE))
scaled_max = int(round((max_rwi + RWI_OFFSET) * RWI_SCALE))

calc_cmd = shutil.which('gdal_calc.py') or shutil.which('gdal_calc')
if not calc_cmd:
    raise RuntimeError('gdal_calc.py not found in PATH')

subprocess.run([
    calc_cmd,
    '-A', str(RWI_COG),
    '--outfile', str(RWI_VALUE_SCALED),
    '--calc', f'((A=={RWI_NODATA})*0) + ((A!={RWI_NODATA})*round((A+{RWI_OFFSET})*{RWI_SCALE}))',
    '--type', 'UInt16',
    '--NoDataValue', '0',
    '--overwrite'
], check=True)

with VALUE_COLORS.open('w', encoding='utf-8') as f:
    f.write('nv 0 0 0 0\n')
    for n in range(max(0, scaled_min), scaled_max + 1):
        r = n & 255
        g = (n >> 8) & 255
        b = (n >> 16) & 255
        f.write(f'{n} {r} {g} {b}\n')

subprocess.run([
    'gdaldem', 'color-relief', '-nearest_color_entry',
    str(RWI_VALUE_SCALED),
    str(VALUE_COLORS),
    str(RWI_VALUE_ENCODED)
], check=True)

TILES_VALUES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'gdal2tiles.py', '-r', 'near', '-z', '6-14', '--xyz', '-w', 'none',
    str(RWI_VALUE_ENCODED), str(TILES_VALUES)
], check=True)
print('Value tiles written to:', TILES_VALUES)
print('Scaled range used for value encoding:', (scaled_min, scaled_max))

In [ ]:
# Write metadata for frontend and provenance
metadata = {
    'dataset_id': 'relative_wealth_index',
    'dataset_name': 'Relative Wealth Index (Brazil)',
    'publisher': 'Meta AI for Good',
    'source_url': 'https://ai.meta.com/ai-for-good/datasets/relative-wealth-index/',
    'resolution': '2.4km',
    'value_encoding': {
        'type': 'rgb_packed',
        'scale': RWI_SCALE,
        'offset': -RWI_OFFSET,
        'unit': RWI_UNIT,
        'decode_formula': 'value = (R + 256*G + 65536*B) / scale + offset'
    },
    'citation': {
        'title': 'Microestimates of wealth for all low- and middle-income countries',
        'authors': [
            'Guanghua Chi',
            'Han Fang',
            'Sourav Chatterjee',
            'Joshua E. Blumenstock'
        ],
        'journal': 'Proceedings of the National Academy of Sciences',
        'year': 2022,
        'volume_issue': '119 (3)',
        'article': 'e2113658119',
        'doi': '10.1073/pnas.2113658119',
        'url': 'https://www.pnas.org/content/119/3/e2113658119'
    },
    'links': {
        'dataset': 'https://ai.meta.com/ai-for-good/datasets/relative-wealth-index/',
        'paper': 'https://www.pnas.org/content/119/3/e2113658119',
        'press': 'https://www.fastcompany.com/90625436/these-new-poverty-maps-could-reshape-how-we-deliver-humanitarian-aid',
        'interactive_map': 'http://beta.povertymaps.net/'
    }
}

with METADATA_JSON.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Metadata written to:', METADATA_JSON)